In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, Trace
from pydantic import BaseModel
from agents.mcp import MCPServerStdio
import os

from IPython.display import display, Markdown, Image
import re

load_dotenv()

True

In [2]:
# System Prompt Templates:

agent1_instructions = ("You are a database analyst creating executive-ready business summaries.\n"
            "Workflow:\n"
            "1) Mandatory discovery first: run execute_query on INFORMATION_SCHEMA.TABLES (BASE TABLE) and INFORMATION_SCHEMA.COLUMNS to find relevant tables/columns by keywords from the user request.\n"
            "2) Use list_tables as a secondary helper only; if it conflicts with SQL discovery, trust INFORMATION_SCHEMA results.\n"
            "3) Inspect top candidate tables with describe_table, then run minimal sample queries (for example SELECT TOP 1) to confirm semantics before final analysis SQL.\n"
            "4) Use execute_query for all analysis and final outputs.\n"
            "5) Use fetch only when external context is needed (for example definitions, benchmarks, market context, or supporting references).\n"
            "6) Adapt to the schema that exists in the current database; do not assume specific table or column names.\n"
            "7) If ideal fields are missing, proceed with best available proxies and clearly state assumptions and limitations instead of stopping.\n"
            "8) Do not ask the user to choose another database or ask permission to continue; proceed autonomously with the current data and provide best-effort findings.\n"
            "9) Do not claim data is unavailable until SQL discovery is complete and at least one relevant analysis query has been attempted.\n"
            "10) Accuracy rule: do not infer trends or rankings from unordered TOP queries; use explicit ORDER BY on the metric being analyzed.\n"
            "11) Output format: keep the main summary concise and business-friendly. Technical detail is allowed in a short 'Technical Notes' section.\n"
            "12) In the main summary, avoid SQL mechanics; include technical specifics only in 'Technical Notes' and 'SQL Appendix'.\n"
            "13) Include key insights, trends, implications, and notable data-quality caveats.\n"
            "14) Where helpful, use available chart tools to generate clear visuals to illustrate comparisons or trends; use text diagrams only if chart tools are unavailable.\n"
            "15) Add a final section titled 'SQL Appendix' and place the exact SQL query there.\n"
            "16) If fetch was used, add a short 'Sources' section with links.")

In [3]:
#MCP Params:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

sql_params={
        "command": "mssql-mcp-server",   # or "npx", "args": ["-y", "mssql-mcp-server"]
        "env": {
            "MSSQL_CONNECTION_STRING": os.getenv("MSSQL_ADV_CONNECTION_STRING"),
            
            # "MSSQL_HOST": os.getenv("MSSQL_HOST"),
            # "MSSQL_PORT": os.getenv("MSSQL_PORT", "1433"),
            # "MSSQL_DATABASE": os.getenv("MSSQL_DATABASE"),
            # "MSSQL_USER": os.getenv("MSSQL_USER"),
            # "MSSQL_PASSWORD": os.getenv("MSSQL_PASSWORD"),
            # "MSSQL_ENCRYPT": os.getenv("MSSQL_ENCRYPT", "false"),
            # "MSSQL_TRUST_SERVER_CERTIFICATE": os.getenv("MSSQL_TRUST_SERVER_CERTIFICATE", "true"),            
        }
}

chart_params = {
      "command": "npx", "args": ["-y", "@antv/mcp-server-chart"]
    }
  

In [4]:
fetch_server = MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60)

mssql_server = MCPServerStdio(params=sql_params, client_session_timeout_seconds=60)

chart_server = MCPServerStdio(params=chart_params, client_session_timeout_seconds=60)

In [5]:
async with mssql_server:
    print(await mssql_server.call_tool("test_connection", {}))

meta=None content=[TextContent(type='text', text='[\n  {\n    "version": "Microsoft SQL Server 2019 (RTM) - 15.0.2000.5 (X64) \\n\\tSep 24 2019 13:48:23 \\n\\tCopyright (C) 2019 Microsoft Corporation\\n\\tExpress Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\\n",\n    "server_name": "LAPTOP-CPI3JAKQ\\\\SQLEXPRESS",\n    "database_name": "AdventureWorks2019"\n  }\n]', annotations=None, meta=None)] structuredContent=None isError=False


In [6]:
async with mssql_server:
    tools = await mssql_server.list_tools()
    print([t.name for t in tools])

['list_connections', 'test_connection', 'list_databases', 'list_tables', 'describe_table', 'sample_data', 'execute_query', 'get_relationships', 'list_stored_procedures', 'describe_stored_procedure', 'list_views', 'describe_view', 'list_triggers', 'describe_trigger', 'list_functions', 'list_indexes', 'analyze_table_stats', 'find_missing_indexes', 'analyze_index_usage', 'analyze_database_size', 'list_constraints', 'analyze_check_constraints', 'list_user_defined_types', 'find_computed_columns', 'list_default_constraints', 'analyze_data_distribution', 'find_lookup_tables', 'analyze_null_patterns', 'detect_audit_columns', 'get_stored_procedure_definition', 'get_multiple_stored_procedure_definitions', 'get_all_stored_procedure_definitions', 'search_stored_procedures_by_content']


In [7]:
async with fetch_server:
    tools = await fetch_server.list_tools()
    print([t.name for t in tools])

['fetch']


In [8]:
async with chart_server:
    tools = await chart_server.list_tools()
    print([t.name for t in tools])

['generate_area_chart', 'generate_bar_chart', 'generate_boxplot_chart', 'generate_column_chart', 'generate_district_map', 'generate_dual_axes_chart', 'generate_fishbone_diagram', 'generate_flow_diagram', 'generate_funnel_chart', 'generate_histogram_chart', 'generate_line_chart', 'generate_liquid_chart', 'generate_mind_map', 'generate_network_graph', 'generate_organization_chart', 'generate_path_map', 'generate_pie_chart', 'generate_pin_map', 'generate_radar_chart', 'generate_sankey_chart', 'generate_scatter_chart', 'generate_treemap_chart', 'generate_venn_chart', 'generate_violin_chart', 'generate_waterfall_chart', 'generate_word_cloud_chart', 'generate_spreadsheet']


In [12]:
async with mssql_server, fetch_server, chart_server:
    agent = Agent(
            name="SQL Analyst",
            #model="gpt-4o-mini",
            model="gpt-5.2",
            instructions=agent1_instructions,
            mcp_servers=[mssql_server, fetch_server, chart_server],
        )

    result = await Runner.run(
        agent,
        max_turns=50,
        input="What customer segments are growing fastest?"
        #input="Select top 10 from salesperson table and show me the results in a tidy markup"
        )
    #print(result.final_output)

    display(Markdown(result.final_output))  # renders markdown tables/images if present

    # optional: if output contains markdown image link, render it explicitly
    m = re.search(r'!\[[^\]]*\]\(([^)]+)\)', result.final_output or "")
    if m:
        display(Image(url=m.group(1)))

## Fastest-growing customer segments (based on active customers ordering)

**Fastest-growing:** **Business (Store)** customers  
- **Approx. monthly growth (CAGR):** **~56.2%**
- **Monthly active customers range (last 12 months of orders):** **2 → 270** (very volatile)
- **Interpretation:** This segment *appears* to be growing the fastest, but the growth rate is heavily influenced by extremely low “floor” months (e.g., 2–3 active customers), which makes percentage growth look dramatic.

**Steadier growth (larger base):** **Consumer (Person)** customers  
- **Approx. monthly growth (CAGR):** **~8.4%**
- **Monthly active customers range (last 12 months of orders):** **898 → 2,171**
- **Interpretation:** Much more stable and substantial in absolute volume; growth is real but not as explosive as the business segment.

### Visual: monthly active customers trend (last 12 months of orders)
![Active customers by month (last 12 months of orders)](https://mdn.alipayobjects.com/one_clip/afts/img/mekRSrNLOMEAAAAAS7AAAAgAoEACAQFr/original)

## Implications
- If you’re prioritizing **growth rate**, **Business (Store)** is the headline winner—but validate whether those very low months reflect **data sparsity, seasonality, or operational reporting artifacts**.
- If you’re prioritizing **scale + consistent momentum**, **Consumer (Person)** is the core growth engine (higher and steadier active-customer counts).

## Data quality / caveats (important)
- “Segments” were inferred from schema as:
  - **Business (Store)** = `Sales.Customer.StoreID IS NOT NULL`
  - **Consumer (Person)** = `Sales.Customer.StoreID IS NULL`
- Growth is computed on **monthly active customers** (distinct customers with ≥1 order), not revenue.
- The last-12-month window is anchored to `MAX(SalesOrderHeader.OrderDate)`; the window includes months where the business segment has **near-zero actives**, inflating growth metrics.

## Technical Notes
- Metric: **monthly active customers** = `COUNT(DISTINCT SalesOrderHeader.CustomerID)` grouped by order month and segment.
- Growth rate shown is an **approx. monthly CAGR** using the min and max monthly values observed within the window (sensitive to outliers).

## SQL Appendix

```sql
SELECT
  s.customer_segment,
  MIN(s.order_month) AS first_month_in_window,
  MAX(s.order_month) AS last_month_in_window,
  AVG(CAST(s.active_customers AS float)) AS avg_monthly_active_customers,
  MIN(s.active_customers) AS min_monthly_active_customers,
  MAX(s.active_customers) AS max_monthly_active_customers,
  CASE WHEN MIN(s.active_customers) = 0 THEN NULL
       ELSE (MAX(s.active_customers) - MIN(s.active_customers)) / CAST(MIN(s.active_customers) AS float) END AS growth_from_min_to_max_pct,
  CASE WHEN MIN(s.active_customers) = 0 THEN NULL
       ELSE POWER(CAST(MAX(s.active_customers) AS float) / NULLIF(MIN(s.active_customers),0), 1.0/11) - 1 END AS approx_monthly_cagr
FROM (
  SELECT
      DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1) AS order_month,
      CASE WHEN c.StoreID IS NULL THEN 'Consumer (Person)' ELSE 'Business (Store)' END AS customer_segment,
      COUNT(DISTINCT soh.CustomerID) AS active_customers
  FROM Sales.SalesOrderHeader soh
  JOIN Sales.Customer c
    ON c.CustomerID = soh.CustomerID
  WHERE soh.OrderDate > (
        SELECT DATEADD(month, -12, MAX(OrderDate))
        FROM Sales.SalesOrderHeader
  )
  GROUP BY DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1),
           CASE WHEN c.StoreID IS NULL THEN 'Consumer (Person)' ELSE 'Business (Store)' END
) s
GROUP BY s.customer_segment
ORDER BY approx_monthly_cagr DESC;
```

```sql
SELECT
  DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1) AS order_month,
  CASE WHEN c.StoreID IS NULL THEN 'Consumer (Person)' ELSE 'Business (Store)' END AS customer_segment,
  COUNT(DISTINCT soh.CustomerID) AS active_customers
FROM Sales.SalesOrderHeader soh
JOIN Sales.Customer c
  ON c.CustomerID = soh.CustomerID
WHERE soh.OrderDate > (
    SELECT DATEADD(month, -12, MAX(OrderDate))
    FROM Sales.SalesOrderHeader
)
GROUP BY DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1),
         CASE WHEN c.StoreID IS NULL THEN 'Consumer (Person)' ELSE 'Business (Store)' END
ORDER BY order_month, customer_segment;
```